In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [ ]:
df = pd.read_csv("processed_data.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes

In [ ]:
df.drop(['User ID','Charging Start Time','Charging End Time','Charging_Start','Charging_end'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
object_cols = df.select_dtypes(include=['object']).columns

In [ ]:
le = LabelEncoder()
for col in object_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
df.head()

#### Train - Test split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop(columns=["Charging Cost (USD)"]) 
y = df["Charging Cost (USD)"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("Training Set Shape:", X_train.shape, y_train.shape)
print("Testing Set Shape:", X_test.shape, y_test.shape)

 train_test_split() ? Splits dataset into training (80%) and testing (20%).

 random_state=42 ? Ensures consistent results each time you run the code.
 
 X_train, X_test, y_train, y_test ? Four datasets for model training & evaluation.

## Model Training and Model Selection

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
def model_evaluation(true,predicted):
    mse = mean_squared_error(true, predicted)
    mae = mean_absolute_error(true,predicted)
    rmse = np.sqrt(mean_squared_error(true,predicted)) 
    r2 = r2_score(true,predicted)
    return mse, mae, r2

## Supervised Regression Models using feature Charging cost (USD)

In [ ]:
models = {
    "Random Forest": RandomForestRegressor(),
    "XGBoost": XGBRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = model_evaluation(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = model_evaluation(y_test, y_test_pred)

    print(list(models.keys())[i])

    print("Models performance on training set")
    print("- Root Mean Squared Error:{:.4f} ".format(model_train_rmse))
    print("- Mean Absolute Error:{:.4f} ".format(model_train_mae))
    print("- R^2:{:.4f} ".format(model_train_r2))

    print("------------------------------------")

    print("Models performance on testing set")
    print("- Root Mean Squared Error:{:.4f} ".format(model_test_rmse))
    print("- Mean Absolute Error:{:.4f} ".format(model_test_mae))
    print("- R^2:{:.4f} ".format(model_test_r2))

    print("="*35)
    print("\n")

RANDOM FOREST

- RMSE (Root Mean Squared Error) and MAE (Mean Absolute Error) are both 0, which means the model perfectly predicts the target variable.
- R? = 1.0000 suggests the model explains 100% of the variance in the data.

XGBoost

- The RMSE and MAE are extremely low, meaning the model makes very small prediction errors.
- R? = 1.0000 again suggests a perfect fit.
- Training RMSE is slightly lower than testing RMSE, indicating that the model generalizes well.

#### Finding best parameters

In [ ]:
rf_params = {
    'n_estimators': [100, 200, 500, 1000],
    'max_features': [5, 10, 'auto', 8],
    'max_depth' : [5, 10, 30, None, 50],
    'min_samples_split': [2, 8, 15, 20],
}

xgb_params = {
    'learning_rate': [0.01, 0.1],
    'max_depth': [ 5, 8, 12, 20, 30],
    'n_estimators': [100, 200, 500],
    'colsample_bytree': [0.5, 0.8, 1, 0.3, 0.4]
}


In [ ]:
randomcv_models = [
    ("RF",RandomForestRegressor(), rf_params),
    ("XGBoost",XGBRegressor(), xgb_params)
]

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

models_best_params = {}
for name, model, params in randomcv_models:
    randomcv = RandomizedSearchCV(estimator=model,
                                param_distributions= params,
                                n_iter=100,
                                cv=3, 
                                n_jobs=-1, 
                                verbose=2, 
                                random_state=42)
    randomcv.fit(X_train, y_train)
    models_best_params[name] = randomcv.best_params_

for model_name in models_best_params:
    print(f'------------ Best parameters for {model_name}------------')
    print(models_best_params[model_name])

In [ ]:
models = {
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_features=8, 
                                                    max_depth=None, min_samples_split=2, n_jobs=-1),       
    "XGBoost Regressor": XGBRegressor(n_estimators=500, max_depth=30, learning_rate=0.1, colsample_bytree=0.3)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = model_evaluation(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = model_evaluation(y_test, y_test_pred)

    print(list(models.keys())[i])

    print("Models performance on training set")
    print("- Root Mean Squared Error:{:.4f} ".format(model_train_rmse))
    print("- Mean Absolute Error:{:.4f} ".format(model_train_mae))
    print("- R^2:{:.4f} ".format(model_train_r2))

    print("------------------------------------")

    print("Models performance on testing set")
    print("- Root Mean Squared Error:{:.4f} ".format(model_test_rmse))
    print("- Mean Absolute Error:{:.4f} ".format(model_test_mae))
    print("- R^2:{:.4f} ".format(model_test_r2))

    print("="*35)
    print("\n")

## Supervised classification model using feature Energy Category

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, auc, f1_score, precision_score, recall_score

In [ ]:
X2 = df.drop(columns=["Energy Category"]) 
y2 = df["Energy Category"]

In [ ]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, random_state=42)

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGBoost": XGBClassifier()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train2, y_train2)

    y_train_pred2 = model.predict(X_train2)
    y_test_pred2 = model.predict(X_test2)

    model_train_accuracy = accuracy_score(y_train2, y_train_pred2)
    model_train_f1 = f1_score(y_train2, y_train_pred2, average='macro') 
    model_train_precision = precision_score(y_train2, y_train_pred2, average='macro')
    model_train_recall = recall_score(y_train2, y_train_pred2, average='macro')
    model_train_roc_auc = roc_auc_score(y_train2, model.predict_proba(X_train2), multi_class='ovr')  

    model_test_accuracy = accuracy_score(y_test2, y_test_pred2)
    model_test_f1 = f1_score(y_test2, y_test_pred2, average='macro')
    model_test_precision = precision_score(y_test2, y_test_pred2, average='macro')
    model_test_recall = recall_score(y_test2, y_test_pred2, average='macro')
    model_test_roc_auc = roc_auc_score(y_test2, model.predict_proba(X_test2), multi_class='ovr')

    print(list(models.keys())[i])

    print("Models performance on training set")
    print("- Accuracy:{:.4f} ".format(model_train_accuracy))
    print("- F1 Score:{:.4f} ".format(model_train_f1))
    print("- Precision:{:.4f} ".format(model_train_precision))
    print("- Recall:{:.4f} ".format(model_train_recall))
    print("- ROC AUC score:{:.4f} ".format(model_train_roc_auc))

    print("------------------------------------")

    print("Models performance on testing set")
    print("- Accuracy:{:.4f} ".format(model_test_accuracy))
    print("- F1 Score:{:.4f} ".format(model_test_f1))
    print("- Precision:{:.4f} ".format(model_test_precision))
    print("- Recall:{:.4f} ".format(model_test_recall))
    print("- ROC AUC score:{:.4f} ".format(model_test_roc_auc))

    print("="*35)
    print("\n")



In [ ]:
adaboost_params = {
    'n_estimators': [50, 10, 20, 50, 100],
    'algorithm': ['SAMME', 'SAMME.R']
}

In [ ]:
randomcv_models = [
    ("AdaBoost",AdaBoostClassifier(), adaboost_params)
]

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

models_best_params = {}
for name, model, params in randomcv_models:
    randomcv = RandomizedSearchCV(estimator=model,
                                param_distributions= params,
                                n_iter=100,
                                cv=3, 
                                n_jobs=-1, 
                                verbose=2, 
                                random_state=42)
    randomcv.fit(X_train2, y_train2)
    models_best_params[name] = randomcv.best_params_

for model_name in models_best_params:
    print(f'------------ Best parameters for {model_name}------------')
    print(models_best_params[model_name])

In [ ]:
models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=50, algorithm='SAMME.R')
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train2, y_train2)

    y_train_pred2 = model.predict(X_train2)
    y_test_pred2 = model.predict(X_test2)

    model_train_accuracy = accuracy_score(y_train2, y_train_pred2)
    model_train_f1 = f1_score(y_train2, y_train_pred2, average='macro') 
    model_train_precision = precision_score(y_train2, y_train_pred2, average='macro')
    model_train_recall = recall_score(y_train2, y_train_pred2, average='macro')
    model_train_roc_auc = roc_auc_score(y_train2, model.predict_proba(X_train2), multi_class='ovr')  

    model_test_accuracy = accuracy_score(y_test2, y_test_pred2)
    model_test_f1 = f1_score(y_test2, y_test_pred2, average='macro')
    model_test_precision = precision_score(y_test2, y_test_pred2, average='macro')
    model_test_recall = recall_score(y_test2, y_test_pred2, average='macro')
    model_test_roc_auc = roc_auc_score(y_test2, model.predict_proba(X_test2), multi_class='ovr')

    print(list(models.keys())[i])

    print("Models performance on training set")
    print("- Accuracy:{:.4f} ".format(model_train_accuracy))
    print("- F1 Score:{:.4f} ".format(model_train_f1))
    print("- Precision:{:.4f} ".format(model_train_precision))
    print("- Recall:{:.4f} ".format(model_train_recall))
    print("- ROC AUC score:{:.4f} ".format(model_train_roc_auc))

    print("------------------------------------")

    print("Models performance on testing set")
    print("- Accuracy:{:.4f} ".format(model_test_accuracy))
    print("- F1 Score:{:.4f} ".format(model_test_f1))
    print("- Precision:{:.4f} ".format(model_test_precision))
    print("- Recall:{:.4f} ".format(model_test_recall))
    print("- ROC AUC score:{:.4f} ".format(model_test_roc_auc))

    print("="*35)
   

As the Adaboost model has lower performace than other we will not use it.

### Saving models into pickle file 

In [ ]:
import pickle
import joblib

#### Regressor models

In [ ]:
rf_regressor = RandomForestRegressor(n_estimators=100, max_features=8, max_depth=None, min_samples_split=2, n_jobs=-1)
rf_regressor.fit(X_train, y_train)

xgb_regressor = XGBRegressor(n_estimators=500, max_depth=30, learning_rate=0.1, colsample_bytree=0.3)
xgb_regressor.fit(X_train, y_train)

In [ ]:
with open("random_forest_regressor.pkl", "wb") as f:
    pickle.dump(rf_regressor, f)

with open("xgboost_regressor.pkl", "wb") as f:
    pickle.dump(xgb_regressor, f)

print("? All models saved successfully!")

In [ ]:

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train2, y_train2)

gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train2, y_train2)

xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train2, y_train2)


In [ ]:


with open("random_forest.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("gradient_boosting.pkl", "wb") as f:
    pickle.dump(gb_model, f)

with open("xgboost.pkl", "wb") as f:
    pickle.dump(xgb_model, f)

print("? Models saved successfully!")
